In [1]:
import pandas as pd
import re
from datasets import Dataset
from sklearn.model_selection import train_test_split

df = pd.read_csv("/kaggle/input/medquad-csv/medquad.csv")

# Extended exploration
print("Dataset Info:")
print(df.info())
print("\nColumns:", df.columns.tolist())
print("\nUnique focus areas:", df['focus_area'].nunique())
print("\nSample questions:", df['question'].head().tolist())
print("\nSample answers:", df['answer'].head().tolist())

# Analyze text lengths
df['question_length'] = df['question'].apply(lambda x: len(str(x).split()))
df['answer_length'] = df['answer'].apply(lambda x: len(str(x).split()))
print("\nQuestion Length Stats:", df['question_length'].describe())
print("Answer Length Stats:", df['answer_length'].describe())

# Refined clean_references function
def clean_references(text):
    if not isinstance(text, str):
        return ""
    
    # Remove contact info
    phone_pattern = r'\b\d{3}-\d{3}-\d{4}\b|\b\d{3}-\d{2}-\d{4}\b|\b1-\d{3}-\d{3}-\d{4}\b|\b\d{4}-\d{3}-\d{4}\b|\bToll Free:.*?\b'
    email_pattern = r'\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Z|a-z]{2,}\b'
    url_pattern = r'http[s]?://(?:[a-zA-Z]|[0-9]|[$-_@.&+]|[!*\\(\\),]|(?:%[0-9a-fA-F][0-9a-fA-F]))+'
    address_pattern = r'\d+\s+[A-Za-z\s]+,\s*[A-Za-z\s]+,\s*[A-Z]{2}\s*\d{5}(-\d{4})?'
    
    text = re.sub(phone_pattern, '', text, flags=re.IGNORECASE)
    text = re.sub(email_pattern, '', text)
    text = re.sub(url_pattern, '', text)
    text = re.sub(address_pattern, '', text)
    
    # Only remove sentences with non-medical reference keywords
    ref_keywords = ['toll free', 'phone', 'email', 'fax', 'tty', 'clearinghouse']
    sentences = text.split('. ')
    cleaned_sentences = [s for s in sentences if not any(keyword.lower() in s.lower() for keyword in ref_keywords)]
    cleaned_text = '. '.join(cleaned_sentences).strip()
    
    # Normalize whitespace
    cleaned_text = re.sub(r'\s+', ' ', cleaned_text)
    cleaned_text = re.sub(r'\.\s*\.', '.', cleaned_text)
    
    return cleaned_text if cleaned_text else text

# Simple medical term normalization (extend with UMLS or dictionary if available)
medical_terms = {'htn': 'hypertension', 'dm': 'diabetes mellitus'}
def normalize_terms(text):
    if not isinstance(text, str):
        return text
    for term, replacement in medical_terms.items():
        text = re.sub(r'\b' + term + r'\b', replacement, text, flags=re.IGNORECASE)
    return text

# Apply preprocessing
df = df.dropna(subset=['answer'])
df['answer'] = df['answer'].apply(clean_references).apply(normalize_terms)
df['question'] = df['question'].apply(normalize_terms).str.strip()
df['focus_area'] = df['focus_area'].fillna('Unknown').str.strip()

# Incorporate focus_area in input_text
def preprocess(example):
    return {
        'input_text': f"answer the medical question (focus: {example['focus_area']}): {example['question']}",
        'target_text': example['answer']
    }

# Split and convert to Datasets
train_df, val_df = train_test_split(df, test_size=0.2, random_state=42)
train_dataset = Dataset.from_pandas(train_df)
val_dataset = Dataset.from_pandas(val_df)
train_dataset = train_dataset.map(preprocess)
val_dataset = val_dataset.map(preprocess)

print(f"Train size: {len(train_dataset)}")
print(f"Validation size: {len(val_dataset)}")

df.info()

Dataset Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16412 entries, 0 to 16411
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   question    16412 non-null  object
 1   answer      16407 non-null  object
 2   source      16412 non-null  object
 3   focus_area  16398 non-null  object
dtypes: object(4)
memory usage: 513.0+ KB
None

Columns: ['question', 'answer', 'source', 'focus_area']

Unique focus areas: 5126

Sample questions: ['What is (are) Glaucoma ?', 'What causes Glaucoma ?', 'What are the symptoms of Glaucoma ?', 'What are the treatments for Glaucoma ?', 'What is (are) Glaucoma ?']

Sample answers: ["Glaucoma is a group of diseases that can damage the eye's optic nerve and result in vision loss and blindness. While glaucoma can strike anyone, the risk is much greater for people over 60. How Glaucoma Develops  There are several different types of glaucoma. Most of these involve the drainage system with

Map:   0%|          | 0/13125 [00:00<?, ? examples/s]

Map:   0%|          | 0/3282 [00:00<?, ? examples/s]

Train size: 13125
Validation size: 3282
<class 'pandas.core.frame.DataFrame'>
Index: 16407 entries, 0 to 16411
Data columns (total 6 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   question         16407 non-null  object
 1   answer           16407 non-null  object
 2   source           16407 non-null  object
 3   focus_area       16407 non-null  object
 4   question_length  16407 non-null  int64 
 5   answer_length    16407 non-null  int64 
dtypes: int64(2), object(4)
memory usage: 897.3+ KB


In [8]:
from transformers import T5Tokenizer, T5ForConditionalGeneration
import re

# Load tokenizer and model
tokenizer = T5Tokenizer.from_pretrained('t5-base')
model = T5ForConditionalGeneration.from_pretrained('t5-base')

# Optional: Analyze text lengths to optimize max_length
def analyze_text_lengths(dataset, field):
    lengths = [len(tokenizer.encode(item)) for item in dataset[field] if isinstance(item, str)]
    print(f"{field} - Mean length: {np.mean(lengths):.2f}, Max length: {max(lengths)}")
    return lengths

# Run analysis (optional, for debugging)
import numpy as np
analyze_text_lengths(train_dataset, 'input_text')
analyze_text_lengths(train_dataset, 'target_text')

# Updated tokenize function
def tokenize(batch):
    # Filter out empty or invalid texts
    inputs = [text for text in batch['input_text'] if isinstance(text, str) and text.strip()]
    targets = [text for text in batch['target_text'] if isinstance(text, str) and text.strip()]
    
    if not inputs or not targets:
        print("Warning: Empty inputs or targets in batch")
        return {'input_ids': [], 'attention_mask': [], 'labels': []}

    # Clean text: remove excessive whitespace, special characters
    inputs = [re.sub(r'\s+', ' ', text).strip() for text in inputs]
    targets = [re.sub(r'\s+', ' ', text).strip() for text in targets]

    # Tokenize with dynamic max_length (adjust based on analysis)
    max_length = 512  # Can reduce to 256 if analysis shows shorter texts
    inputs_enc = tokenizer(
        inputs,
        padding='max_length',
        truncation=True,
        max_length=max_length,
        return_tensors='pt',
        return_attention_mask=True
    )
    labels_enc = tokenizer(
        targets,
        padding='max_length',
        truncation=True,
        max_length=max_length,
        return_tensors='pt'
    )

    return {
        'input_ids': inputs_enc['input_ids'].tolist(),
        'attention_mask': inputs_enc['attention_mask'].tolist(),
        'labels': labels_enc['input_ids'].tolist()
    }

# Apply tokenization
try:
    train_dataset = train_dataset.map(tokenize, batched=True, remove_columns=['input_text', 'target_text'])
    val_dataset = val_dataset.map(tokenize, batched=True, remove_columns=['input_text', 'target_text'])
except Exception as e:
    print(f"Tokenization error: {e}")
    raise

# Set dataset format for PyTorch
train_dataset.set_format('torch', columns=['input_ids', 'attention_mask', 'labels'])
val_dataset.set_format('torch', columns=['input_ids', 'attention_mask', 'labels'])

input_text - Mean length: 33.60, Max length: 89
target_text - Mean length: 297.71, Max length: 6185


Map:   0%|          | 0/13125 [00:00<?, ? examples/s]

Map:   0%|          | 0/3282 [00:00<?, ? examples/s]

In [14]:
from transformers import T5Tokenizer, T5ForConditionalGeneration, Trainer, TrainingArguments

tokenizer = T5Tokenizer.from_pretrained('t5-base')
model = T5ForConditionalGeneration.from_pretrained('t5-base')

def tokenize(batch):
    inputs = tokenizer(batch['input_text'], padding='max_length', truncation=True, max_length=512)
    labels = tokenizer(batch['target_text'], padding='max_length', truncatiType 1 diabetes is a condition thaton=True, max_length=512)
    inputs['labels'] = labels['input_ids']
    return inputs

train_dataset = train_dataset.map(tokenize, batched=True)
val_dataset = val_dataset.map(tokenize, batched=True)

training_args = TrainingArguments(
    output_dir='./results',
    run_name="t5_finetune_medical",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    warmup_steps=500,
    weight_decay=0.01,
    logging_dir='./logs',
    eval_strategy="epoch",   
    save_strategy="epoch",
    load_best_model_at_end=True,
    logging_strategy="steps",     
    logging_steps=50,             
    report_to="none",             
)



trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer
)

trainer.train()

# Save model
model.save_pretrained('./fine_tuned_t5_medical')
tokenizer.save_pretrained('./fine_tuned_t5_medical')


/tmp/ipykernel_36/3825469194.py:53: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
Passing a tuple of `past_key_values` is deprecated and will be removed in Transformers v4.48.0. You should pass an instance of `EncoderDecoderCache` instead, e.g. `past_key_values=EncoderDecoderCache.from_legacy_cache(past_key_values)`.
/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:70: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Epoch,Training Loss,Validation Loss


Training error: CUDA out of memory. Tried to allocate 5.15 GiB. GPU 0 has a total capacity of 14.74 GiB of which 4.91 GiB is free. Process 3821 has 9.83 GiB memory in use. Of the allocated memory 7.79 GiB is allocated by PyTorch, and 1.58 GiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)


In [1]:
!pip install evaluate
!pip install rouge_score

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 6.7 MB/s eta 0:00:00
^C
ERROR: Operation cancelled by user
  Preparing metadata (setup.py) ... done
  Created wheel for rouge_score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=2aec4972fe4bad63176250ec002f3f8af552feddb3b99d93e3c83036548b0603
  Stored in directory: /root/.cache/pip/wheels/1e/19/43/8a442dc83660ca25e163e1bd1f89919284ab0d0c1475475148
Successfully built rouge_score


In [24]:
!zip -r /kaggle/working/fine_tuned_t5_medical.zip ./fine_tuned_t5_medical


updating: fine_tuned_t5_medical/ (stored 0%)
updating: fine_tuned_t5_medical/config.json (deflated 63%)
updating: fine_tuned_t5_medical/model.safetensors

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


 (deflated 8%)
updating: fine_tuned_t5_medical/generation_config.json (deflated 29%)
updating: fine_tuned_t5_medical/added_tokens.json (deflated 83%)
updating: fine_tuned_t5_medical/special_tokens_map.json (deflated 85%)
updating: fine_tuned_t5_medical/tokenizer_config.json (deflated 94%)
updating: fine_tuned_t5_medical/spiece.model (deflated 48%)


In [26]:
!pip install huggingface_hub
from huggingface_hub import notebook_login
notebook_login()


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [27]:
from huggingface_hub import create_repo

create_repo("fine_tuned_t5_medical", private=False)  # or private=True if you want


RepoUrl('https://huggingface.co/moazzamisdead/fine_tuned_t5_medical', endpoint='https://huggingface.co', repo_type='model', repo_id='moazzamisdead/fine_tuned_t5_medical')

In [29]:
from huggingface_hub import upload_folder

upload_folder(
    folder_path="./fine_tuned_t5_medical",
    repo_id="moazzamisdead/fine_tuned_t5_medical",
    repo_type="model"
)


Uploading...:   0%|          | 0.00/892M [00:00<?, ?B/s]

CommitInfo(commit_url='https://huggingface.co/moazzamisdead/fine_tuned_t5_medical/commit/63fc13f816fb099b54ab1ef2a74fb13248dde8ea', commit_message='Upload folder using huggingface_hub', commit_description='', oid='63fc13f816fb099b54ab1ef2a74fb13248dde8ea', pr_url=None, repo_url=RepoUrl('https://huggingface.co/moazzamisdead/fine_tuned_t5_medical', endpoint='https://huggingface.co', repo_type='model', repo_id='moazzamisdead/fine_tuned_t5_medical'), pr_revision=None, pr_num=None)

In [2]:
!pip install sentence-transformers faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.4/31.4 MB 61.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 104.3 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 74.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 38.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 2.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 31.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 13.3 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 8.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 84.4 MB/s eta 0:00:00:00:0100:01
  Attempting un

In [3]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
import faiss
import pickle

df = pd.read_csv("/kaggle/input/medquad-csv/medquad.csv")
df = df.dropna(subset=['answer']) 

embedder = SentenceTransformer('all-MiniLM-L6-v2')

print("Generating embeddings for answers...")
answer_texts = df['answer'].tolist()
embeddings = embedder.encode(answer_texts, batch_size=32, show_progress_bar=True)

embeddings = np.array(embeddings, dtype='float32')

dimension = embeddings.shape[1] 
index = faiss.IndexFlatL2(dimension)  # L2 distance for similarity search
index.add(embeddings)  # Add embeddings to the index

# Save the FAISS index and answer metadata for later use
faiss.write_index(index, './medquad_faiss_index.bin')
with open('./medquad_answer_metadata.pkl', 'wb') as f:
    pickle.dump({'answer_texts': answer_texts, 'df': df}, f)

print("FAISS index and metadata saved successfully.")

2025-08-22 14:56:41.896694: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1755874602.261027      36 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1755874602.368616      36 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Generating embeddings for answers...


Batches:   0%|          | 0/513 [00:00<?, ?it/s]

FAISS index and metadata saved successfully.


In [4]:
import numpy as np
from sentence_transformers import SentenceTransformer
import faiss
import pickle
import pandas as pd


index = faiss.read_index('./medquad_faiss_index.bin')
with open('./medquad_answer_metadata.pkl', 'rb') as f:
    metadata = pickle.load(f)
answer_texts = metadata['answer_texts']
df = metadata['df']

# Load sentence-transformers model (same as used for indexing)
embedder = SentenceTransformer('all-MiniLM-L6-v2')

def retrieve_context(query, k=5):
    """
    Retrieve top-k relevant contexts for a given query.
    
    Args:
        query (str): The user query.
        k (int): Number of contexts to retrieve.
    
    Returns:
        List of dictionaries containing question, answer, and focus_area for top-k matches.
    """
    # Embed the query
    query_embedding = embedder.encode([query], batch_size=1)
    query_embedding = np.array(query_embedding, dtype='float32')
    
    # Search FAISS index for top-k similar answers
    distances, indices = index.search(query_embedding, k)
    
    # Retrieve corresponding question-answer pairs
    contexts = []
    for idx in indices[0]:
        context = {
            'question': df.iloc[idx]['question'],
            'answer': df.iloc[idx]['answer'],
            'focus_area': df.iloc[idx]['focus_area']
        }
        contexts.append(context)
    
    return contexts

# Example usage (for testing)
if __name__ == "__main__":
    sample_query = "What is the treatment for pressure?"
    contexts = retrieve_context(sample_query, k=5)
    print("Retrieved Contexts:")
    for i, ctx in enumerate(contexts, 1):
        print(f"\nContext {i}:")
        print(f"Question: {ctx['question']}")
        print(f"Answer: {ctx['answer'][:200]}...")  # Truncate for display
        print(f"Focus Area: {ctx['focus_area']}")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Retrieved Contexts:

Context 1:
Question: What are the treatments for Melkersson-Rosenthal Syndrome ?
Answer: Treatment is symptomatic and may include medication therapies with nonsteroidal anti-inflammatory drugs (NSAIDs) and corticosteroids to reduce swelling, as well as antibiotics and immunosuppressants. ...
Focus Area: Melkersson-Rosenthal Syndrome

Context 2:
Question: What are the treatments for Hemifacial Spasm ?
Answer: Surgical treatment in the form of microvascular decompression, which relieves pressure on the facial nerve, will relieve hemifacial spasm in many cases. This intervention has significant potential sid...
Focus Area: Hemifacial Spasm

Context 3:
Question: What are the treatments for Achalasia ?
Answer: How might achalasia be treated? The aim of treatment is to reduce the pressure at the lower esophageal sphincter. Therapy may involve: Injection with botulinum toxin (Botox) to help relax the sphincte...
Focus Area: Achalasia

Context 4:
Question: What are the tre

In [7]:
import numpy as np
from sentence_transformers import SentenceTransformer
import faiss
import pickle
import pandas as pd
from transformers import T5Tokenizer, T5ForConditionalGeneration
import torch

# Load FAISS index and metadata
try:
    index = faiss.read_index('./medquad_faiss_index.bin')
    with open('./medquad_answer_metadata.pkl', 'rb') as f:
        metadata = pickle.load(f)
    answer_texts = metadata['answer_texts']
    df = metadata['df']
except FileNotFoundError as e:
    print(f"Error: {e}. Ensure Step 1 was executed and files are in /kaggle/working/.")
    raise

# Load sentence-transformers model for embeddings
embedder = SentenceTransformer('all-MiniLM-L6-v2')

def retrieve_context(query, k=3):
    """
    Retrieve top-k relevant contexts for a given query.
    
    Args:
        query (str): The user query.
        k (int): Number of contexts to retrieve.
    
    Returns:
        List of dictionaries containing question, answer, and focus_area for top-k matches.
    """
    # Embed the query
    query_embedding = embedder.encode([query], batch_size=1)
    query_embedding = np.array(query_embedding, dtype='float32')
    
    # Search FAISS index for top-k similar answers
    distances, indices = index.search(query_embedding, k)
    
    # Retrieve corresponding question-answer pairs
    contexts = []
    for idx in indices[0]:
        context = {
            'question': df.iloc[idx]['question'],
            'answer': df.iloc[idx]['answer'][:500],  # Limit answer length to avoid overflow
            'focus_area': df.iloc[idx]['focus_area']
        }
        contexts.append(context)
    
    return contexts

# Load fine-tuned T5 model and tokenizer
try:
    tokenizer = T5Tokenizer.from_pretrained('moazzamisdead/fine_tuned_t5_medical')
    model = T5ForConditionalGeneration.from_pretrained('moazzamisdead/fine_tuned_t5_medical')
except FileNotFoundError as e:
    print(f"Error: {e}. Ensure the fine-tuned model was saved correctly in ./fine_tuned_t5_medical.")
    raise

# Move model to GPU if available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)
model.eval()

def generate_answer(query, k=3, max_length=512, max_new_tokens=150):
    """
    Generate an answer for a query using retrieved contexts and fine-tuned T5 model.
    
    Args:
        query (str): The user query.
        k (int): Number of contexts to retrieve.
        max_length (int): Max input length for tokenization.
        max_new_tokens (int): Max tokens to generate.
    
    Returns:
        str: Generated answer or error message if generation fails.
    """
    # Retrieve contexts
    contexts = retrieve_context(query, k=k)
    
    # Format contexts into a concise string
    context_str = ""
    for idx, ctx in enumerate(contexts, 1):
        context_str += f"[{idx}] {ctx['answer']} "
    context_str = context_str[:1000]  # Limit context length to prevent overflow
    
    # Create input prompt for T5
    input_text = f"Using the following medical information: {context_str} Answer the question: {query}"
    
    # Tokenize input
    try:
        inputs = tokenizer(
            input_text,
            padding='max_length',
            truncation=True,
            max_length=max_length,
            return_tensors='pt'
        )
    except Exception as e:
        print(f"Tokenization error: {e}")
        return "Error: Unable to process the query."
    
    # Move inputs to the same device as the model
    inputs = {key: val.to(device) for key, val in inputs.items()}
    
    # Generate answer
    try:
        outputs = model.generate(
            input_ids=inputs['input_ids'],
            attention_mask=inputs['attention_mask'],
            max_length=max_new_tokens,
            num_beams=5,
            no_repeat_ngram_size=3,  # Prevent repetitive phrases
            length_penalty=0.8,      # Slightly favor shorter outputs
            early_stopping=True
        )
        answer = tokenizer.decode(outputs[0], skip_special_tokens=True).strip()
    except Exception as e:
        print(f"Generation error: {e}")
        return "Error: Unable to generate an answer."
    
    # Check if answer is valid
    if not answer or len(answer.split()) < 3:
        return "Error: Generated answer is too short or invalid."
    
    return answer

# Example usage (for testing)
if __name__ == "__main__":
    sample_query = "What is the sugar disease?"
    answer = generate_answer(sample_query)
    print(f"Query: {sample_query}")
    print(f"Generated Answer: {answer}")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Query: What is the sugar disease?
Generated Answer: HFI is caused by mutations in the ALDOB gene. It is inherited in an autosomal recessive pattern. People with HFI have high blood glucose, which is a form of glucose that enters the bloodstream. The body uses glucose to break down sugars and starches in the body. This process is called glucocorticoidosis. Insulin helps the body break down glucose into glucose, a substance that is found in the blood. Glucocortisone is produced by the body's cells and is produced in the liver. The glucose produced from fructose and sucrose are stored in cells called glucose oxidas


In [10]:
import numpy as np
from sentence_transformers import SentenceTransformer
import faiss
import pickle
import pandas as pd
from transformers import T5Tokenizer, T5ForConditionalGeneration
import torch
import re

# Load the FAISS index and metadata
index = faiss.read_index('./medquad_faiss_index.bin')
with open('./medquad_answer_metadata.pkl', 'rb') as f:
    metadata = pickle.load(f)
answer_texts = metadata['answer_texts']
df = metadata['df']

# Load sentence-transformers model for embeddings
embedder = SentenceTransformer('all-MiniLM-L6-v2')

def retrieve_context(query, k=5):
    """
    Retrieve top-k relevant contexts for a given query.
    
    Args:
        query (str): The user query.
        k (int): Number of contexts to retrieve.
    
    Returns:
        List of dictionaries containing question, answer, and focus_area for top-k matches.
    """
    # Embed the query
    query_embedding = embedder.encode([query], batch_size=1)
    query_embedding = np.array(query_embedding, dtype='float32')
    
    # Search FAISS index for top-k similar answers
    distances, indices = index.search(query_embedding, k)
    
    # Retrieve corresponding question-answer pairs
    contexts = []
    for idx in indices[0]:
        context = {
            'question': df.iloc[idx]['question'],
            'answer': df.iloc[idx]['answer'],
            'focus_area': df.iloc[idx]['focus_area']
        }
        contexts.append(context)
    
    return contexts

# Load fine-tuned T5 model and tokenizer
tokenizer = T5Tokenizer.from_pretrained('moazzamisdead/fine_tuned_t5_medical')
model = T5ForConditionalGeneration.from_pretrained('moazzamisdead/fine_tuned_t5_medical')

# Move model to GPU if available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)
model.eval()

def medical_qa_bot(query, k=3, max_input_length=512, max_new_tokens=200):
    """
    Medical QA bot that retrieves relevant contexts and generates an answer.
    
    Args:
        query (str): The user query.
        k (int): Number of contexts to retrieve.
        max_input_length (int): Max input length for tokenization.
        max_new_tokens (int): Max tokens to generate.
    
    Returns:
        dict: Contains the generated answer and retrieved contexts.
    """
    # Input validation
    if not isinstance(query, str) or not query.strip():
        return {"error": "Invalid or empty query", "answer": None, "contexts": []}
    
    # Clean query
    query = re.sub(r'\s+', ' ', query).strip()
    
    # Retrieve contexts
    try:
        contexts = retrieve_context(query, k=k)
    except Exception as e:
        return {"error": f"Retrieval failed: {str(e)}", "answer": None, "contexts": []}
    
    # Format contexts for input (truncate answers to avoid exceeding max_input_length)
    context_str = ""
    for idx, ctx in enumerate(contexts, 1):
        # Truncate answer to 100 tokens to keep input manageable
        answer_tokens = tokenizer.encode(ctx['answer'], max_length=100, truncation=True)
        truncated_answer = tokenizer.decode(answer_tokens, skip_special_tokens=True)
        context_str += f"Context {idx}: Question: {ctx['question']} Answer: {truncated_answer} "
    
    # Create input prompt
    input_text = f"Using the following contexts, answer the medical question: {context_str}Query: {query}"
    
    # Debug: Print input prompt length
    input_tokens = tokenizer.encode(input_text)
    print(f"Input prompt token length: {len(input_tokens)}")
    
    # Tokenize input
    try:
        inputs = tokenizer(
            input_text,
            padding='max_length',
            truncation=True,
            max_length=max_input_length,
            return_tensors='pt'
        )
    except Exception as e:
        return {"error": f"Tokenization failed: {str(e)}", "answer": None, "contexts": contexts}
    
    # Move inputs to the same device as the model
    inputs = {key: val.to(device) for key, val in inputs.items()}
    
    # Generate answer
    try:
        outputs = model.generate(
            input_ids=inputs['input_ids'],
            attention_mask=inputs['attention_mask'],
            max_new_tokens=max_new_tokens,  # Explicitly set new tokens
            num_beams=5,
            no_repeat_ngram_size=3,  # Prevent repetitive n-grams
            length_penalty=0.8,  # Slightly penalize longer outputs
            early_stopping=True
        )
        answer = tokenizer.decode(outputs[0], skip_special_tokens=True).strip()
    except Exception as e:
        return {"error": f"Generation failed: {str(e)}", "answer": None, "contexts": contexts}
    
    return {
        "answer": answer,
        "contexts": contexts,
        "error": None
    }

# Example usage (for testing)
if __name__ == "__main__":
    sample_query = "What is the backbone pain?"
    result = medical_qa_bot(sample_query)
    print(f"Query: {sample_query}")
    if result["error"]:
        print(f"Error: {result['error']}")
    else:
        print(f"Answer: {result['answer']}")
        print("\nRetrieved Contexts:")
        for i, ctx in enumerate(result['contexts'], 1):
            print(f"\nContext {i}:")
            print(f"Question: {ctx['question']}")
            print(f"Answer: {ctx['answer'][:200]}...")  # Truncate for display
            print(f"Focus Area: {ctx['focus_area']}")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Input prompt token length: 347
Query: What is the backbone pain?
Answer: Your backbone is made up of 26 bones called vertebrae. In between them are soft disks filled with a jelly-like substance. These disks cushion your spine and keep it in place. As you age, the disks break down or degenerate. As the discs break, they lose their cushioning ability. A herniated disk is a disk that ruptures. This can lead to pain in the spine and other parts of the body. Symptoms of Paget's disease of bone include - Pain in the legs, skull, and spine. - Muscle weakness, weakness, and scoliosis can occur in people with Pagets disease.

Retrieved Contexts:

Context 1:
Question: What is (are) Spine Injuries and Disorders ?
Answer: Your backbone, or spine, is made up of 26 bone discs called vertebrae. The vertebrae protect your spinal cord and allow you to stand and bend. A number of problems can change the structure of the spin...
Focus Area: Spine Injuries and Disorders

Context 2:
Question: What is (are)

In [11]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
import faiss
import pickle
from transformers import T5Tokenizer, T5ForConditionalGeneration
import torch
import re
from datasets import load_dataset
from rouge_score import rouge_scorer
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
import nltk

# Download NLTK data for BLEU
nltk.download('punkt')

# Load the FAISS index and metadata
try:
    index = faiss.read_index('./medquad_faiss_index.bin')
    with open('./medquad_answer_metadata.pkl', 'rb') as f:
        metadata = pickle.load(f)
    answer_texts = metadata['answer_texts']
    df = metadata['df']
except Exception as e:
    print(f"Error loading FAISS index or metadata: {e}")
    raise

# Load sentence-transformers model for embeddings
embedder = SentenceTransformer('all-MiniLM-L6-v2')

def retrieve_context(query, k=5):
    """
    Retrieve top-k relevant contexts for a given query.
    
    Args:
        query (str): The user query.
        k (int): Number of contexts to retrieve.
    
    Returns:
        List of dictionaries containing question, answer, and focus_area for top-k matches.
    """
    query_embedding = embedder.encode([query], batch_size=1)
    query_embedding = np.array(query_embedding, dtype='float32')
    distances, indices = index.search(query_embedding, k)
    contexts = []
    for idx in indices[0]:
        context = {
            'question': df.iloc[idx]['question'],
            'answer': df.iloc[idx]['answer'],
            'focus_area': df.iloc[idx]['focus_area']
        }
        contexts.append(context)
    return contexts

# Load fine-tuned T5 model and tokenizer
try:
    tokenizer = T5Tokenizer.from_pretrained('moazzamisdead/fine_tuned_t5_medical')
    model = T5ForConditionalGeneration.from_pretrained('moazzamisdead/fine_tuned_t5_medical')
except Exception as e:
    print(f"Error loading T5 model or tokenizer: {e}")
    raise

# Move model to GPU if available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)
model.eval()

def medical_qa_bot(query, k=5, max_input_length=512, max_new_tokens=200):
    """
    Medical QA bot that retrieves relevant contexts and generates an answer.
    
    Args:
        query (str): The user query.
        k (int): Number of contexts to retrieve.
        max_input_length (int): Max input length for tokenization.
        max_new_tokens (int): Max tokens to generate.
    
    Returns:
        dict: Contains the generated answer and retrieved contexts.
    """
    if not isinstance(query, str) or not query.strip():
        return {"error": "Invalid or empty query", "answer": None, "contexts": []}
    
    query = re.sub(r'\s+', ' ', query).strip()
    
    try:
        contexts = retrieve_context(query, k=k)
    except Exception as e:
        return {"error": f"Retrieval failed: {str(e)}", "answer": None, "contexts": []}
    
    context_str = ""
    for idx, ctx in enumerate(contexts, 1):
        context_str += f"Context {idx}: {ctx['question']} Answer: {ctx['answer']} "
    
    input_text = f"answer the medical question using context: {context_str} query: {query}"
    
    try:
        inputs = tokenizer(
            input_text,
            padding='max_length',
            truncation=True,
            max_length=max_input_length,
            return_tensors='pt'
        )
    except Exception as e:
        return {"error": f"Tokenization failed: {str(e)}", "answer": None, "contexts": contexts}
    
    inputs = {key: val.to(device) for key, val in inputs.items()}
    
    try:
        outputs = model.generate(
            input_ids=inputs['input_ids'],
            attention_mask=inputs['attention_mask'],
            max_length=max_new_tokens,
            num_beams=5,
            length_penalty=1.0,
            early_stopping=True
        )
        answer = tokenizer.decode(outputs[0], skip_special_tokens=True).strip()
    except Exception as e:
        return {"error": f"Generation failed: {str(e)}", "answer": None, "contexts": contexts}
    
    return {
        "answer": answer,
        "contexts": contexts,
        "error": None
    }

# Load TREC-2017 LiveQA medical test set
try:
    test_data = load_dataset('hyesunyun/liveqa_medical_trec2017', split='test')
    print("Dataset columns:", test_data.column_names)
    # Inspect first few entries
    print("Sample data (first 3 rows):")
    for i in range(min(3, len(test_data))):
        print(f"Row {i+1}:")
        print(f"Question: {test_data[i]['ORIGINAL_QUESTION_MESSAGE']}")
        print(f"Reference Answer: {test_data[i]['REFERENCE_ANSWERS']}")
    
    # Preprocess dataset
    def preprocess_answer(answer):
        if isinstance(answer, list):
            # Extract ANSWER field from each dictionary and join valid strings
            answers = [item['ANSWER'] for item in answer if isinstance(item, dict) and 'ANSWER' in item and isinstance(item['ANSWER'], str) and item['ANSWER'].strip()]
            return " ".join(answers).strip()
        elif isinstance(answer, str):
            return answer.strip()
        else:
            return ""
    
    test_df = pd.DataFrame({
        'question': [str(q).strip() if isinstance(q, str) else "" for q in test_data['ORIGINAL_QUESTION_MESSAGE']],
        'reference_answer': [preprocess_answer(a) for a in test_data['REFERENCE_ANSWERS']]
    })
    
    # Filter out invalid entries
    test_df = test_df[
        (test_df['question'].str.strip() != '') &
        (test_df['reference_answer'].str.strip() != '')
    ]
    print(f"Number of valid questions after preprocessing: {len(test_df)}")
except Exception as e:
    print(f"Could not load or process TREC-2017 LiveQA: {e}")
    print("Falling back to MedQuAD subset for evaluation...")
    medquad_df = pd.read_csv("/kaggle/input/medquad-csv/medquad.csv")
    medquad_df = medquad_df.dropna(subset=['question', 'answer'])
    test_df = medquad_df.sample(n=100, random_state=42)[['question', 'answer']]
    test_df.columns = ['question', 'reference_answer']
    print(f"Using {len(test_df)} MedQuAD samples as fallback")

# Take a subset of up to 100 valid test questions
test_df = test_df.head(100)

# Initialize metrics
scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
smoothing = SmoothingFunction().method1
results = []

# Evaluate the QA bot
print("Evaluating QA bot on test set...")
for idx, row in test_df.iterrows():
    query = row['question']
    reference = row['reference_answer']
    
    # Run the QA bot
    result = medical_qa_bot(query, k=5)
    
    if result['error']:
        print(f"Error on question {idx+1}: {result['error']}")
        continue
    
    generated_answer = result['answer']
    
    # Compute ROUGE scores
    rouge_scores = scorer.score(reference, generated_answer)
    
    # Compute BLEU score
    reference_tokens = [nltk.word_tokenize(reference)]
    generated_tokens = nltk.word_tokenize(generated_answer)
    bleu_score = sentence_bleu(reference_tokens, generated_tokens, smoothing_function=smoothing)
    
    # Store results
    results.append({
        'question': query,
        'reference_answer': reference,
        'generated_answer': generated_answer,
        'rouge1': rouge_scores['rouge1'].fmeasure,
        'rouge2': rouge_scores['rouge2'].fmeasure,
        'rougeL': rouge_scores['rougeL'].fmeasure,
        'bleu': bleu_score
    })

# Convert results to DataFrame
results_df = pd.DataFrame(results)

# Compute average metrics
avg_metrics = {
    'avg_rouge1': results_df['rouge1'].mean() if not results_df.empty else 0.0,
    'avg_rouge2': results_df['rouge2'].mean() if not results_df.empty else 0.0,
    'avg_rougeL': results_df['rougeL'].mean() if not results_df.empty else 0.0,
    'avg_bleu': results_df['bleu'].mean() if not results_df.empty else 0.0
}

# Print summary
print("\nEvaluation Summary:")
print(f"Average ROUGE-1 F1: {avg_metrics['avg_rouge1']:.4f}")
print(f"Average ROUGE-2 F1: {avg_metrics['avg_rouge2']:.4f}")
print(f"Average ROUGE-L F1: {avg_metrics['avg_rougeL']:.4f}")
print(f"Average BLEU: {avg_metrics['avg_bleu']:.4f}")

# Save results to CSV
results_df.to_csv('/kaggle/working/evaluation_results.csv', index=False)
print("Detailed results saved to '/kaggle/working/evaluation_results.csv'")

# Display sample results
print("\nSample Results (first 5):")
for idx, row in results_df.head(5).iterrows():
    print(f"\nQuestion {idx+1}: {row['question']}")
    print(f"Reference Answer: {row['reference_answer'][:200]}...")
    print(f"Generated Answer: {row['generated_answer'][:200]}...")
    print(f"ROUGE-1: {row['rouge1']:.4f}, ROUGE-2: {row['rouge2']:.4f}, ROUGE-L: {row['rougeL']:.4f}, BLEU: {row['bleu']:.4f}")

[nltk_data] Downloading package punkt to /usr/share/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


README.md: 0.00B [00:00, ?B/s]

test.jsonl: 0.00B [00:00, ?B/s]

Generating test split:   0%|          | 0/104 [00:00<?, ? examples/s]

Dataset columns: ['NIST_PARAPHRASE', 'NLM_SUMMARY', 'REFERENCE_ANSWERS', 'QUESTION_ID', 'ORIGINAL_QUESTION_SUBJECT', 'ORIGINAL_QUESTION_MESSAGE', 'ORIGINAL_QUESTION_FILE', 'ANNOTATIONS_FOCUS', 'ANNOTATIONS_TYPE', 'ANNOTATIONS_KEYWORD']
Sample data (first 3 rows):
Row 1:
Question: What are the references with noonan syndrome and polycystic renal disease
Reference Answer: [{'ANSWER': "Noonan's syndrome is an eponymic designation that has been used during the last 8 years to describe a variable constellation of somatic and visceral congenital anomalies, which includes groups of patients previously referred to as male Turner's, female pseudo-Turner's and\n\t\t\t\t\tBonnevie-Ullrich syndromes. It is now recognized that both sexes may show the stigmas of this condition and, unlike Turner's syndrome, there is no karyotype abnormality although there is often a familial pattern. The most commonly observed anomalies include webbing of the neck, hypertelorism, a\n\t\t\t\t\tshield-shaped chest and

...

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]


Evaluation Summary:
Average ROUGE-1 F1: 0.1565
Average ROUGE-2 F1: 0.0235
Average ROUGE-L F1: 0.1047
Average BLEU: 0.0090
Detailed results saved to '/kaggle/working/evaluation_results.csv'

Sample Results (first 5):

Question 1: What are the references with noonan syndrome and polycystic renal disease
Reference Answer: Noonan's syndrome is an eponymic designation that has been used during the last 8 years to describe a variable constellation of somatic and visceral congenital anomalies, which includes groups of pati...
Generated Answer: Noonan syndrome is a genetic disorder that causes abnormal development of multiple parts of the body. It occurs in approximately 1 in 1,000 to 2,500 people. It is caused by mutations in any one of sev...
ROUGE-1: 0.2291, ROUGE-2: 0.0374, ROUGE-L: 0.1300, BLEU: 0.0059

Question 2: Re:NDC# 0115-0672-50 Zolmitriptan tabkets 5mg. I have celiac disease & need to know if these contain gluten, Thank you!
Reference Answer: Zolmitriptan tablets are available as

In [39]:
# Add to the end of evaluate_qa_bot.py
print("\nDetailed Comparison for Low-Scoring Questions (ROUGE-1 < 0.2):")
low_score_df = results_df[results_df['rouge1'] < 0.2]
for idx, row in low_score_df.head(5).iterrows():
    print(f"\nQuestion {idx+1}: {row['question']}")
    print(f"Reference Answer: {row['reference_answer'][:500]}...")
    print(f"Generated Answer: {row['generated_answer'][:500]}...")
    print(f"ROUGE-1: {row['rouge1']:.4f}, BLEU: {row['bleu']:.4f}")


Detailed Comparison for Low-Scoring Questions (ROUGE-1 < 0.2):

Question 2: Re:NDC# 0115-0672-50 Zolmitriptan tabkets 5mg. I have celiac disease & need to know if these contain gluten, Thank you!
Reference Answer: Zolmitriptan tablets are available as 2.5 mg (yellow and functionally-scored) and 5 mg (pink, not scored) film coated tablets for oral administration. The film coated tablets contain anhydrous lactose NF, microcrystalline cellulose NF, sodium starch glycolate NF, magnesium stearate NF,
					hydroxypropyl methylcellulose USP, titanium dioxide USP, polyethylene glycol 400 NF, yellow iron oxide NF (2.5 mg tablet), red iron oxide NF (5 mg tablet), and polyethylene glycol 8000 NF.
					Zolmitriptan o...
Generated Answer: Celiac disease is an immune disease in which people can't eat gluten because it will damage their small intestine. It is an immune disease in which people can't eat gluten because it will damage their small intestine. Celiac disease is an immune disease in which 

In [40]:
# Add to the end of evaluate_qa_bot.py
print("\nDetailed Comparison for Sample Questions:")
for idx, row in results_df.head(5).iterrows():
    print(f"\nQuestion {idx+1}: {row['question']}")
    print(f"Reference Answer: {row['reference_answer'][:500]}...")
    print(f"Generated Answer: {row['generated_answer'][:500]}...")
    print(f"ROUGE-1: {row['rouge1']:.4f}, BLEU: {row['bleu']:.4f}")


Detailed Comparison for Sample Questions:

Question 1: What are the references with noonan syndrome and polycystic renal disease
Reference Answer: Noonan's syndrome is an eponymic designation that has been used during the last 8 years to describe a variable constellation of somatic and visceral congenital anomalies, which includes groups of patients previously referred to as male Turner's, female pseudo-Turner's and
					Bonnevie-Ullrich syndromes. It is now recognized that both sexes may show the stigmas of this condition and, unlike Turner's syndrome, there is no karyotype abnormality although there is often a familial pattern. The most ...
Generated Answer: Noonan syndrome is a genetic disorder that causes abnormal development of multiple parts of the body. It occurs in approximately 1 in 1,000 to 2,500 people. It is caused by mutations in any one of several genes including the PTPN11, KRAS, RAF1, SOS1, NRAS and BRAF genes. It is sometimes referred to as a specific subtype based on

In [49]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
import faiss
import pickle
from transformers import T5Tokenizer, T5ForConditionalGeneration
import torch
import re
from datasets import load_dataset
from rouge_score import rouge_scorer
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
import nltk
try:
    from bert_score import score as bert_score
except ImportError:
    print("Installing bert-score...")
    import os
    os.system("pip install bert-score")
    from bert_score import score as bert_score

# Download NLTK data for BLEU
nltk.download('punkt')

# Load the FAISS index and metadata
try:
    index = faiss.read_index('./medquad_faiss_index.bin')
    with open('./medquad_answer_metadata.pkl', 'rb') as f:
        metadata = pickle.load(f)
    answer_texts = metadata['answer_texts']
    df = metadata['df']
except Exception as e:
    print(f"Error loading FAISS index or metadata: {e}")
    raise

# Load sentence-transformers model for embeddings
embedder = SentenceTransformer('all-MiniLM-L6-v2')

def retrieve_context(query, k=5):
    """
    Retrieve top-k relevant contexts for a given query.
    
    Args:
        query (str): The user query.
        k (int): Number of contexts to retrieve.
    
    Returns:
        List of dictionaries containing question, answer, and focus_area for top-k matches.
    """
    query_embedding = embedder.encode([query], batch_size=1)
    query_embedding = np.array(query_embedding, dtype='float32')
    distances, indices = index.search(query_embedding, k)
    contexts = []
    for idx in indices[0]:
        context = {
            'question': df.iloc[idx]['question'],
            'answer': df.iloc[idx]['answer'],
            'focus_area': df.iloc[idx]['focus_area']
        }
        contexts.append(context)
    return contexts

# Load fine-tuned T5 model and tokenizer
try:
    tokenizer = T5Tokenizer.from_pretrained('moazzamisdead/fine_tuned_t5_medical')
    model = T5ForConditionalGeneration.from_pretrained('moazzamisdead/fine_tuned_t5_medical')
except Exception as e:
    print(f"Error loading T5 model or tokenizer: {e}")
    raise

# Move model to GPU if available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)
model.eval()

def medical_qa_bot(query, k=5, max_input_length=512, max_new_tokens=300):
    """
    Medical QA bot that retrieves relevant contexts and generates an answer.
    
    Args:
        query (str): The user query.
        k (int): Number of contexts to retrieve.
        max_input_length (int): Max input length for tokenization.
        max_new_tokens (int): Max tokens to generate.
    
    Returns:
        dict: Contains the generated answer and retrieved contexts.
    """
    if not isinstance(query, str) or not query.strip():
        return {"error": "Invalid or empty query", "answer": None, "contexts": []}
    
    query = re.sub(r'\s+', ' ', query).strip()
    
    try:
        contexts = retrieve_context(query, k=k)
    except Exception as e:
        return {"error": f"Retrieval failed: {str(e)}", "answer": None, "contexts": []}
    
    context_str = ""
    for idx, ctx in enumerate(contexts, 1):
        context_str += f"Context {idx}: {ctx['question']} Answer: {ctx['answer']} "
    
    input_text = f"Provide a detailed medical answer using context: {context_str} query: {query}"
    
    try:
        inputs = tokenizer(
            input_text,
            padding='max_length',
            truncation=True,
            max_length=max_input_length,
            return_tensors='pt'
        )
    except Exception as e:
        return {"error": f"Tokenization failed: {str(e)}", "answer": None, "contexts": contexts}
    
    inputs = {key: val.to(device) for key, val in inputs.items()}
    
    try:
        outputs = model.generate(
            input_ids=inputs['input_ids'],
            attention_mask=inputs['attention_mask'],
            max_length=max_new_tokens,
            num_beams=5,
            length_penalty=1.0,
            early_stopping=True
        )
        answer = tokenizer.decode(outputs[0], skip_special_tokens=True).strip()
    except Exception as e:
        return {"error": f"Generation failed: {str(e)}", "answer": None, "contexts": contexts}
    
    return {
        "answer": answer,
        "contexts": contexts,
        "error": None
    }

# Load TREC-2017 LiveQA medical test set
try:
    test_data = load_dataset('hyesunyun/liveqa_medical_trec2017', split='test')
    print("Dataset columns:", test_data.column_names)
    # Inspect first few entries
    print("Sample data (first 3 rows):")
    for i in range(min(3, len(test_data))):
        print(f"Row {i+1}:")
        print(f"Question: {test_data[i]['ORIGINAL_QUESTION_MESSAGE']}")
        print(f"Reference Answer: {test_data[i]['REFERENCE_ANSWERS']}")
    
    # Preprocess dataset
    def preprocess_answer(answer):
        if isinstance(answer, list):
            # Extract ANSWER field from each dictionary and join valid strings
            answers = [item['ANSWER'] for item in answer if isinstance(item, dict) and 'ANSWER' in item and isinstance(item['ANSWER'], str) and item['ANSWER'].strip()]
            return " ".join(answers).strip()
        elif isinstance(answer, str):
            return answer.strip()
        else:
            return ""
    
    test_df = pd.DataFrame({
        'question': [str(q).strip() if isinstance(q, str) else "" for q in test_data['ORIGINAL_QUESTION_MESSAGE']],
        'reference_answer': [preprocess_answer(a) for a in test_data['REFERENCE_ANSWERS']]
    })
    
    # Filter out invalid entries
    test_df = test_df[
        (test_df['question'].str.strip() != '') &
        (test_df['reference_answer'].str.strip() != '')
    ]
    print(f"Number of valid questions after preprocessing: {len(test_df)}")
except Exception as e:
    print(f"Could not load or process TREC-2017 LiveQA: {e}")
    print("Falling back to MedQuAD subset for evaluation...")
    medquad_df = pd.read_csv("/kaggle/input/medquad-csv/medquad.csv")
    medquad_df = medquad_df.dropna(subset=['question', 'answer'])
    test_df = medquad_df.sample(n=100, random_state=42)[['question', 'answer']]
    test_df.columns = ['question', 'reference_answer']
    print(f"Using {len(test_df)} MedQuAD samples as fallback")

# Take a subset of up to 100 valid test questions
test_df = test_df.head(100)

# Initialize metrics
scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
smoothing = SmoothingFunction().method1
results = []

# Evaluate the QA bot
print("Evaluating QA bot on test set...")
for idx, row in test_df.iterrows():
    query = row['question']
    reference = row['reference_answer']
    
    # Run the QA bot
    result = medical_qa_bot(query, k=5)
    
    if result['error']:
        print(f"Error on question {idx+1}: {result['error']}")
        continue
    
    generated_answer = result['answer']
    
    # Compute ROUGE scores
    rouge_scores = scorer.score(reference, generated_answer)
    
    # Compute BLEU score
    reference_tokens = [nltk.word_tokenize(reference)]
    generated_tokens = nltk.word_tokenize(generated_answer)
    bleu_score = sentence_bleu(reference_tokens, generated_tokens, smoothing_function=smoothing)
    
    # Compute BERTScore
    try:
        P, R, F1 = bert_score([generated_answer], [reference], lang="en", model_type="bert-base-uncased", verbose=False)
        bertscore_f1 = F1.item()
    except Exception as e:
        print(f"BERTScore failed for question {idx+1}: {e}")
        bertscore_f1 = 0.0
    
    # Store results
    results.append({
        'question': query,
        'reference_answer': reference,
        'generated_answer': generated_answer,
        'rouge1': rouge_scores['rouge1'].fmeasure,
        'rouge2': rouge_scores['rouge2'].fmeasure,
        'rougeL': rouge_scores['rougeL'].fmeasure,
        'bleu': bleu_score,
        'bertscore_f1': bertscore_f1
    })

# Convert results to DataFrame
results_df = pd.DataFrame(results)

# Compute average metrics
avg_metrics = {
    'avg_rouge1': results_df['rouge1'].mean() if not results_df.empty else 0.0,
    'avg_rouge2': results_df['rouge2'].mean() if not results_df.empty else 0.0,
    'avg_rougeL': results_df['rougeL'].mean() if not results_df.empty else 0.0,
    'avg_bleu': results_df['bleu'].mean() if not results_df.empty else 0.0,
    'avg_bertscore_f1': results_df['bertscore_f1'].mean() if not results_df.empty else 0.0
}

# Print summary
print("\nEvaluation Summary:")
print(f"Average ROUGE-1 F1: {avg_metrics['avg_rouge1']:.4f}")
print(f"Average ROUGE-2 F1: {avg_metrics['avg_rouge2']:.4f}")
print(f"Average ROUGE-L F1: {avg_metrics['avg_rougeL']:.4f}")
print(f"Average BLEU: {avg_metrics['avg_bleu']:.4f}")
print(f"Average BERTScore F1: {avg_metrics['avg_bertscore_f1']:.4f}")

# Save results to CSV
results_df.to_csv('/kaggle/working/evaluation_results.csv', index=False)
print("Detailed results saved to '/kaggle/working/evaluation_results.csv'")

# Display sample results
print("\nSample Results (first 5):")
for idx, row in results_df.head(5).iterrows():
    print(f"\nQuestion {idx+1}: {row['question']}")
    print(f"Reference Answer: {row['reference_answer'][:200]}...")
    print(f"Generated Answer: {row['generated_answer'][:200]}...")
    print(f"ROUGE-1: {row['rouge1']:.4f}, ROUGE-2: {row['rouge2']:.4f}, ROUGE-L: {row['rougeL']:.4f}, BLEU: {row['bleu']:.4f}, BERTScore F1: {row['bertscore_f1']:.4f}")

# Display low-scoring examples for inspection
print("\nDetailed Comparison for Low-Scoring Questions (ROUGE-1 < 0.2):")
low_score_df = results_df[results_df['rouge1'] < 0.2]
for idx, row in low_score_df.head(5).iterrows():
    print(f"\nQuestion {idx+1}: {row['question']}")
    print(f"Reference Answer: {row['reference_answer'][:500]}...")
    print(f"Generated Answer: {row['generated_answer'][:500]}...")
    print(f"ROUGE-1: {row['rouge1']:.4f}, BLEU: {row['bleu']:.4f}, BERTScore F1: {row['bertscore_f1']:.4f}")

[nltk_data] Downloading package punkt to /usr/share/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


Dataset columns: ['NIST_PARAPHRASE', 'NLM_SUMMARY', 'REFERENCE_ANSWERS', 'QUESTION_ID', 'ORIGINAL_QUESTION_SUBJECT', 'ORIGINAL_QUESTION_MESSAGE', 'ORIGINAL_QUESTION_FILE', 'ANNOTATIONS_FOCUS', 'ANNOTATIONS_TYPE', 'ANNOTATIONS_KEYWORD']
Sample data (first 3 rows):
Row 1:
Question: What are the references with noonan syndrome and polycystic renal disease
Reference Answer: [{'ANSWER': "Noonan's syndrome is an eponymic designation that has been used during the last 8 years to describe a variable constellation of somatic and visceral congenital anomalies, which includes groups of patients previously referred to as male Turner's, female pseudo-Turner's and\n\t\t\t\t\tBonnevie-Ullrich syndromes. It is now recognized that both sexes may show the stigmas of this condition and, unlike Turner's syndrome, there is no karyotype abnormality although there is often a familial pattern. The most commonly observed anomalies include webbing of the neck, hypertelorism, a\n\t\t\t\t\tshield-shaped chest and

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

...

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]


Evaluation Summary:
Average ROUGE-1 F1: 0.1426
Average ROUGE-2 F1: 0.0226
Average ROUGE-L F1: 0.0984
Average BLEU: 0.0088
Average BERTScore F1: 0.4694
Detailed results saved to '/kaggle/working/evaluation_results.csv'

Sample Results (first 5):

Question 1: What are the references with noonan syndrome and polycystic renal disease
Reference Answer: Noonan's syndrome is an eponymic designation that has been used during the last 8 years to describe a variable constellation of somatic and visceral congenital anomalies, which includes groups of pati...
Generated Answer: Noonan syndrome occurs in approximately 1 in 1,000 to 2,500 people. It is caused by mutations in any one of several genes including the PTPN11, KRAS, RAF1, SOS1, NRAS and BRAF genes. It is sometimes r...
ROUGE-1: 0.2299, ROUGE-2: 0.0269, ROUGE-L: 0.1283, BLEU: 0.0097, BERTScore F1: 0.4856

Question 2: Re:NDC# 0115-0672-50 Zolmitriptan tabkets 5mg. I have celiac disease & need to know if these contain gluten, Thank you!
Refe

In [12]:
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

# Load a medical-specific embedder
embedder = SentenceTransformer('sentence-transformers/paraphrase-mpnet-base-v2')  # Alternative: 'microsoft/biomednlp-pubmedbert-base-uncased'

# Create embeddings with question + focus_area + answer
combined_texts = df.apply(lambda row: f"{row['question']} Focus: {row['focus_area']} {row['answer']}", axis=1).tolist()
combined_embeddings = embedder.encode(combined_texts, convert_to_numpy=True, show_progress_bar=True)

# Initialize FAISS index
dimension = combined_embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(combined_embeddings)l

# Updated retrieve_context function
def retrieve_context(query, k=5):  # Increased k
    try:
        query_embedding = embedder.encode([query], convert_to_numpy=True)
        distances, indices = index.search(query_embedding, k)
        retrieved_contexts = [combined_texts[idx] for idx in indices[0]]
        return retrieved_contexts
    except Exception as e:
        print(f"Retrieval error for query '{query[:50]}...': {e}")
        return []

# Test retrieval quality
sample_queries = ["What are the symptoms of diabetes?", "How is hypertension treated?"]
for query in sample_queries:
    contexts = retrieve_context(query, k=5)
    print(f"\nQuery: {query}")
    print("Retrieved Contexts:")
    for i, ctx in enumerate(contexts, 1):
        print(f"Context {i}: {ctx[:100]}...")

Batches:   0%|          | 0/513 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]


Query: What are the symptoms of diabetes?
Retrieved Contexts:
Context 1: What are the symptoms of Diabetes ? Focus: Diabetes Many people with diabetes experience one or more...
Context 2: What are the symptoms of Diabetes ? Focus: Diabetes Diabetes is often called a "silent" disease beca...
Context 3: What are the symptoms of Your Guide to Diabetes: Type 1 and Type 2 ? Focus: Your Guide to Diabetes: ...
Context 4: What are the symptoms of Prevent diabetes problems: Keep your kidneys healthy ? Focus: Prevent diabe...
Context 5: What are the symptoms of Metabolic Syndrome ? Focus: Metabolic Syndrome Metabolic syndrome is a grou...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]


Query: How is hypertension treated?
Retrieved Contexts:
Context 1: What are the treatments for High Blood Pressure ? Focus: High Blood Pressure Based on your diagnosis...
Context 2: What is (are) Blood Pressure Medicines ? Focus: Blood Pressure Medicines High blood pressure, also c...
Context 3: What are the treatments for High Blood Pressure ? Focus: High Blood Pressure Today, many different t...
Context 4: What are the treatments for Pulmonary arterial hypertension ? Focus: Pulmonary arterial hypertension...
Context 5: How to prevent High Blood Pressure ? Focus: High Blood Pressure Healthy lifestyle habits, proper use...


In [29]:
query = "What is water?"
contexts = retrieve_context(query, k=5)
answer = generate_answer(query, contexts)
print(f"Question: {query}")
print(f"Answer: {answer}")
print("Contexts:", [ctx[:100] + "..." for ctx in contexts])

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Question: What is water?
Answer: WaterhouseFriderichsen syndrome is a disorder of water balance. It is caused by bleeding into the adrenal gland. It is usually caused by severe meningococcal infection or other severe, bacterial infection. WaterhouseFriderichsen syndrome is caused by bleeding into the adrenal gland. People with this condition can become dehydrated if they do not drink enough water. WaterhouseFriderichsen syndrome is caused by bleeding into the adrenal gland. WaterhouseFriderichsen syndrome is caused by bleeding into the adrenal gland. WaterhouseFriderichsen syndrome is caused by bleeding into the adrenal gland. WaterhouseFriderichsen syndrome is caused by bleeding into the adrenal gland. WaterhouseFriderichsen syndrome is caused by bleeding into the adrenal gland. WaterhouseFriderichsen syndrome is caused by bleeding into the adrenal gland. WaterhouseFriderichsen
Contexts: ['What is (are) WaterhouseFriderichsen syndrome ? Focus: WaterhouseFriderichsen syndrome Waterhous

In [26]:
from datasets import load_dataset
import evaluate

ds = load_dataset("hyesunyun/liveqa_medical_trec2017")
test_dataset = ds['test']

test_questions = [example['ORIGINAL_QUESTION_MESSAGE'] for example in test_dataset]
references = [[ans['ANSWER'] for ans in example['REFERENCE_ANSWERS']] for example in test_dataset]

predictions = []
for question in test_questions:
    try:
        contexts = retrieve_context(question, k=3)  # Your existing retrieve_context function
        pred_answer = generate_answer(question, contexts)  # Your existing generate_answer function
        predictions.append(pred_answer)
    except Exception as e:
        print(f"Error generating answer for question '{question[:50]}...': {e}")
        predictions.append("")  # Append empty string to maintain alignment

rouge = evaluate.load("rouge")

try:
    results = rouge.compute(
        predictions=predictions,
        references=references,
        use_aggregator=True,
        use_stemmer=True
    )
except Exception as e:
    print(f"Error computing ROUGE scores: {e}")
    results = {'rouge1': 0.0, 'rouge2': 0.0, 'rougeL': 0.0}

# Print evaluation results
print("Evaluation Results:")
print(f"ROUGE-1: {results['rouge1']:.4f}")
print(f"ROUGE-2: {results['rouge2']:.4f}")
print(f"ROUGE-L: {results['rougeL']:.4f}")

# Print sample outputs (first 5, formatted cleanly)
for i, (question, pred, refs) in enumerate(zip(test_questions[:5], predictions[:5], references[:5])):
    print(f"\nSample {i+1}:")
    print(f"Question: {question}")
    print(f"Predicted Answer: {pred}")
    print(f"Reference Answers:")
    print("\n".join(f"- {ref}" for ref in refs))  
print("\nBERTScore Results:")
print(f"Precision: {np.mean(bert_results['precision']):.4f}")
print(f"Recall: {np.mean(bert_results['recall']):.4f}")
print(f"F1: {np.mean(bert_results['f1']):.4f}")

# Print sample outputs
for i, (q, pred, refs) in enumerate(zip(test_questions[:5], predictions[:5], references[:5])):
    print(f"\nSample {i+1}:")
    print(f"Question: {q}")
    print(f"Predicted Answer: {pred}")
    print("Reference Answers:")
    print("\n".join(f"- {ref}" for ref in refs))microsoft/biomednlp-pubmedbert-base-uncased"
    )
except Exception as e:
    print(f"Error computing BERTScore: {e}")
    bert_results = {'precision': [0.0], 'recall': [0.0], 'f1': [0.0]}

# Compute retrieval precision
retrieval_precision = precision_score(retrieval_labels, retrieval_predictions, zero_division=0)
retrieval_recall = recall_score(retrieval_labels, retrieval_predictions, zero_division=0)
answer_accuracy = np.mean(answer_correctness)

# Print evaluation results
print("\nEvaluation Results:")
print(f"Retrieval Precision: {retrieval_precision:.4f}")
print(f"Retrieval Recall: {retrieval_recall:.4f}")
print(f"Answer Accuracy: {answer_accuracy:.4f}")
print(f"ROUGE-1: {rouge_results['rouge1']:.4f}")
print(f"ROUGE-2: {rouge_results['rouge2']:.4f}")
print(f"ROUGE-L: {rouge_results['rougeL']:.4f}")
print(f"BERTScore Precision: {np.mean(bert_results['precision']):.4f}")
print(f"BERTScore Recall: {np.mean(bert_results['recall']):.4f}")
print(f"BERTScore F1: {np.mean(bert_results['f1']):.4f}")

# Print sample outputs with contexts
for i, (q, pred, refs) in enumerate(zip(test_questions_subset[:5], predictions[:5], references_subset[:5])):
    print(f"\nSample {i+1}:")
    print(f"Question: {q}")
    print(f"Predicted Answer: {pred}")
    print("Reference Answers:")
    print("\n".join(f"- {ref}" for ref in refs))
    contexts = retrieve_context(q, k=5)
    print("Retrieved Contexts:")
    for j, ctx in enumerate(contexts, 1):
        print(f"Context {j}: {ctx[:100]}...")

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

added_tokens.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

KeyboardInterrupt: 